# Regression metrics (MSE, MAE, R²)

Regression metrics choose which kind of error should be expensive.

MSE magnifies large misses, MAE keeps error in original units, and R² compares against predicting the mean. This notebook computes the formulas once, then tracks MSE up the regression ladder. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

np.random.seed(7)


def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))

    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))

    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))

    diabetes = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", diabetes.data, diabetes.target))

    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))

    return rungs


def reg_rmse(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return float(np.sqrt(mean_squared_error(y_te, preds)))


def linear_baseline(x_tr, y_tr, x_te):
    model = LinearRegression()
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def lesson_score(losses, cost, alternative):
    raw = round(float(np.mean(losses)), 3)
    score = round(raw + cost, 3)
    gap = round(alternative - score, 3)
    relative_gap = gap / alternative
    stabilized = 0.8 * score
    return {
        "raw": raw,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stabilized": stabilized,
    }


def preview_reg_ladder(rungs):
    for name, X, y in rungs:
        print(f"{name}: X={X.shape}, y-range=({float(np.min(y)):.3f}, {float(np.max(y)):.3f}), sample={np.round(X[:2], 3).tolist()}")


## The concept, built once on D1

The lesson formula is $$MSE=\frac1m\sum_i e_i^2,\quad MAE=\frac1m\sum_i |e_i|,\quad R^2=1-\frac{SS_{res}}{SS_{tot}}$$. We first rebuild the exact lesson arithmetic before using a reusable method on larger data.

In [ ]:

def regression_metrics_mse_mae_r_method(losses, cost, alternative):
    """Recompute the lesson raw average, cost-aware score, and alternative gap."""
    losses = np.asarray(losses, dtype=float)
    summary = lesson_score(losses, cost, alternative)
    return summary


losses = np.array([0.213, 0.07, 0.488], dtype=float)
summary = regression_metrics_mse_mae_r_method(losses, cost=0.080, alternative=0.389)

assert abs(summary["raw"] - 0.257) < 1e-12
assert abs(summary["score"] - 0.337) < 1e-12
assert abs(summary["gap"] - 0.052) < 1e-12
assert abs(summary["relative_gap"] - 0.134) < 0.001
assert abs(summary["stabilized"] - 0.270) < 0.001

print("losses:", losses.tolist())
print("raw average:", round(summary["raw"], 3))
print("score with cost:", round(summary["score"], 3))
print("gap to alternative:", round(summary["gap"], 3))
print("relative gap:", round(summary["relative_gap"], 3))


The same score object also carries the stabilized launch number and, for this topic, the topic-specific metric calculation used on the toy D1 counts or residuals.

In [ ]:

def regression_metric_values(y_true, y_pred):
    residuals = np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float)
    mse = float(np.mean(residuals ** 2))
    mae = float(np.mean(np.abs(residuals)))
    ss_res = float(np.sum(residuals ** 2))
    ss_tot = float(np.sum((np.asarray(y_true, dtype=float) - np.mean(y_true)) ** 2))
    r2 = 1.0 - ss_res / ss_tot
    return mse, mae, r2

y_true = np.array([3.0, -0.5, 2.0, 7.0])
y_pred = np.array([2.5, 0.0, 2.0, 8.0])
mse, mae, r2 = regression_metric_values(y_true, y_pred)
assert round(mse, 4) == 0.375
assert round(mae, 4) == 0.5
assert round(r2, 4) == 0.9486


print("toy MSE:", round(mse, 4))
print("toy MAE:", round(mae, 4))
print("toy R^2:", round(r2, 4))


## The dataset ladder

The notebook embeds the shared `reg_ladder()` source so it is self-contained in Colab. D5 is a noisy high-dimensional regression problem where metric choice changes model preference.

In [ ]:

rungs = reg_ladder()
preview_reg_ladder(rungs)


## Run MSE across D1–D5

The same linear baseline is fit on every rung and MSE is tracked as the primary metric.

In [ ]:

def mse_rung(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=3)
    model = LinearRegression()
    model.fit(x_tr, y_tr)
    preds = model.predict(x_te)
    mse = float(mean_squared_error(y_te, preds))
    mae = float(mean_absolute_error(y_te, preds))
    rmse = reg_rmse(linear_baseline, X, y)
    return mse, mae, rmse, y_te, preds

results = []
for name, X, y in rungs:
    mse_value, mae_value, rmse_value, y_te, preds = mse_rung(X, y)
    results.append({"name": name, "mse": mse_value, "mae": mae_value, "rmse": rmse_value, "y_te": y_te, "preds": preds})
    print(f"{name}: MSE={mse_value:.3f}, MAE={mae_value:.3f}, reg_rmse={rmse_value:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, row in zip(axes[0], results):
    residuals = row["y_te"] - row["preds"]
    ax.scatter(row["preds"], residuals, s=16, color="purple", alpha=0.75)
    ax.axhline(0.0, color="gray", linewidth=1)
    ax.set_title(row["name"].split(" (")[0], fontsize=8)
axes[1, 0].plot(range(1, 6), [row["mse"] for row in results], marker="o")
axes[1, 0].set_title("MSE vs rung")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("MSE")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

A flexible regressor can reduce raw MSE while increasing operational risk. The lesson score adds the cost before selecting.

In [ ]:

name, X, y = rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=3)
rows = []
for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model.fit(x_tr, y_tr)
    preds = model.predict(x_te)
    raw_mse = float(mean_squared_error(y_te, preds))
    scaled_mse = raw_mse / float(np.var(y_te))
    complexity_cost = 0.080 / (1.0 + alpha)
    decision_score = scaled_mse + complexity_cost
    rows.append((alpha, scaled_mse, complexity_cost, decision_score))

raw_best = min(rows, key=lambda row: row[1])
fixed_best = min(rows, key=lambda row: row[3])
print("D5 candidate table: alpha, scaled MSE, cost, decision score")
for row in rows:
    print(tuple(round(value, 4) for value in row))
print("wrong raw-only choice:", raw_best)
print("cost-aware choice:", fixed_best)
print("decision-score improvement:", round(raw_best[3] - fixed_best[3], 4))
assert fixed_best[3] <= raw_best[3] + 1e-12


## Evaluate it + Practice

- Track `MSE` against a no-skill baseline before trusting the result.
- Sanity-check that D1 reproduces the lesson numbers exactly and that D5 is not a toy shortcut.
- Ablation: choose by R² while ignoring absolute error; the metric should get worse or the selected model should become less stable.
- Failure signals: large validation gap, unstable threshold/criterion, or a cost-aware score that disagrees with the raw metric.
- Report both the raw metric and the cost-aware decision score when the lesson formula includes a cost.

Practice prompts:
1. Change one candidate hyperparameter and recompute the D5 table.
2. Replace the no-skill baseline and explain whether `MSE` moved for the right reason.
3. Add one diagnostic plot that would catch the named pitfall for regression metrics.